# OCR – CCCD (Căn Cước Công Dân / ID Card)

**Pipeline:** Load → Preprocess → OCR → Parse & Structure → Confidence & Save

Mỗi bước in kết quả trung gian để dễ debug.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path
import re
import unicodedata

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from paddleocr import PaddleOCR

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "extract":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

ID_CARD_ROOT = PROJECT_ROOT / "data" / "unstructured" / "documents" / "doc_type=id_card"
run_dirs = sorted(ID_CARD_ROOT.glob("run_date=*"))
INPUT_DIR = run_dirs[-1] if run_dirs else ID_CARD_ROOT / "run_date=2026-05-25"
RUN_DATE = INPUT_DIR.name.split("=", 1)[1]

_n = input("Số lượng ảnh cần xử lý (Enter = tất cả): ").strip()
LIMIT = int(_n) if _n else 0
PREVIEW_N = min(3, LIMIT) if LIMIT else 3
OCR_LANG = "en"

img_paths = sorted(p for ext in ("*.jpg", "*.jpeg", "*.png", "*.tif", "*.tiff")
                   for p in INPUT_DIR.rglob(ext))
if LIMIT:
    img_paths = img_paths[:LIMIT]

print(f"INPUT_DIR : {INPUT_DIR}")
print(f"RUN_DATE  : {RUN_DATE}")
print(f"Images    : {len(img_paths)}  (limit={LIMIT or 'all'})")
print(f"OCR_LANG  : {OCR_LANG}")
pd.DataFrame({"image_path": [str(p) for p in img_paths]}).head(20)

In [ ]:
# ── Image utilities ───────────────────────────────────────────────────────────

def show_bgr(title: str, bgr: np.ndarray, max_w: int = 9) -> None:
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if bgr.ndim == 3 else bgr
    h, w = rgb.shape[:2]
    plt.figure(figsize=(max_w, max_w * h / max(1, w)))
    plt.imshow(rgb, cmap="gray" if rgb.ndim == 2 else None)
    plt.title(title); plt.axis("off"); plt.show()

def normalize_width(bgr: np.ndarray, out_w: int = 1000) -> np.ndarray:
    h, w = bgr.shape[:2]
    if w == out_w:
        return bgr
    out_h = max(1, int(h * out_w / max(1, w)))
    return cv2.resize(bgr, (out_w, out_h),
                      interpolation=cv2.INTER_AREA if out_w < w else cv2.INTER_CUBIC)

def extract_user_id(path: Path) -> int | None:
    for part in path.parts:
        if part.startswith("user_id="):
            try: return int(part.split("=", 1)[1])
            except Exception: pass
    return None


# ── Deskew ────────────────────────────────────────────────────────────────────

def _rotate_bound(bgr: np.ndarray, angle: float, border: tuple = (255, 255, 255)) -> np.ndarray:
    h, w = bgr.shape[:2]
    cx, cy = w / 2.0, h / 2.0
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    new_w, new_h = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += new_w / 2.0 - cx
    M[1, 2] += new_h / 2.0 - cy
    return cv2.warpAffine(bgr, M, (new_w, new_h), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=border)

def _skew_from_edges(edges: np.ndarray, min_len: int) -> float | None:
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=50,
                            minLineLength=min_len, maxLineGap=20)
    if lines is None:
        return None
    angles, weights = [], []
    for x1, y1, x2, y2 in lines[:, 0]:
        angle = float(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
        angle = ((angle + 90.0) % 180.0) - 90.0
        if angle > 45.0: angle -= 90.0
        elif angle < -45.0: angle += 90.0
        if abs(angle) <= 30.0:
            angles.append(angle)
            weights.append(float(np.hypot(x2 - x1, y2 - y1)))
    if not angles:
        return None
    order = np.argsort(angles)
    cumsum = np.cumsum(np.asarray(weights, dtype=np.float32)[order])
    return float(np.asarray(angles, dtype=np.float32)[order][
        int(np.searchsorted(cumsum, cumsum[-1] * 0.5))])

def _estimate_skew(bgr: np.ndarray) -> float | None:
    h, w = bgr.shape[:2]
    min_len = max(80, int(min(h, w) * 0.25))
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    blue_mask = cv2.inRange(hsv, np.array([85, 45, 20]), np.array([135, 255, 190]))
    blue_mask = cv2.morphologyEx(blue_mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8), iterations=2)
    angle = _skew_from_edges(cv2.Canny(blue_mask, 50, 150), max(40, int(min(h, w) * 0.18)))
    if angle is not None:
        return angle
    gray = cv2.GaussianBlur(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    edges = cv2.dilate(cv2.Canny(gray, 50, 150), np.ones((3, 3), np.uint8), iterations=1)
    return _skew_from_edges(edges, min_len)

def deskew(bgr: np.ndarray, min_abs: float = 0.75) -> tuple[np.ndarray, float]:
    angle = _estimate_skew(bgr)
    if angle is None or abs(angle) < min_abs:
        return bgr, 0.0
    candidates = [(0.0, bgr, angle)]
    for rot in (-angle, angle):
        rotated = _rotate_bound(bgr, rot)
        residual = _estimate_skew(rotated)
        candidates.append((rot, rotated, residual if residual is not None else 999.0))
    best_rot, best_img, _ = min(candidates, key=lambda t: abs(t[2]))
    return best_img, float(best_rot)


# ── Document detection & warp ─────────────────────────────────────────────────

def _order_points(pts: np.ndarray) -> np.ndarray:
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1)
    rect[0], rect[2] = pts[np.argmin(s)], pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1], rect[3] = pts[np.argmin(diff)], pts[np.argmax(diff)]
    return rect

def _is_card_quad(pts: np.ndarray, area_img: float) -> bool:
    rect = _order_points(pts)
    tl, tr, br, bl = rect
    w = max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl))
    h = max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl))
    aspect = max(w, h) / max(1.0, min(w, h))
    return cv2.contourArea(pts) / max(1.0, area_img) >= 0.25 and 1.25 <= aspect <= 1.95

def _find_quad(edges: np.ndarray, area_img: float,
               offset: tuple[int, int] = (0, 0)) -> np.ndarray | None:
    ox, oy = offset
    cnts, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    for c in sorted(cnts, key=cv2.contourArea, reverse=True)[:40]:
        approx = cv2.approxPolyDP(c, 0.02 * cv2.arcLength(c, True), True)
        if len(approx) == 4:
            pts = approx.reshape(4, 2).astype(np.float32)
            pts[:, 0] -= ox; pts[:, 1] -= oy
            if _is_card_quad(pts, area_img):
                return _order_points(pts)
    for c in sorted(cnts, key=cv2.contourArea, reverse=True)[:40]:
        if cv2.contourArea(c) <= 0:
            continue
        box = cv2.boxPoints(cv2.minAreaRect(c)).astype(np.float32)
        box[:, 0] -= ox; box[:, 1] -= oy
        if _is_card_quad(box, area_img):
            return _order_points(box)
    return None

def find_document_quad(bgr: np.ndarray) -> np.ndarray | None:
    h, w = bgr.shape[:2]
    pad = max(20, int(min(h, w) * 0.04))
    padded = cv2.copyMakeBorder(bgr, pad, pad, pad, pad,
                                cv2.BORDER_CONSTANT, value=(255, 255, 255))
    gray = cv2.GaussianBlur(cv2.cvtColor(padded, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    edges = cv2.dilate(
        cv2.morphologyEx(cv2.Canny(gray, 35, 120), cv2.MORPH_CLOSE,
                         np.ones((9, 9), np.uint8), iterations=2),
        np.ones((3, 3), np.uint8), iterations=1)
    quad = _find_quad(edges, h * w, offset=(pad, pad))
    if quad is not None:
        return quad
    gray = cv2.GaussianBlur(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    edges = cv2.dilate(cv2.Canny(gray, 50, 150), np.ones((3, 3), np.uint8), iterations=1)
    return _find_quad(edges, h * w)

def warp_document(bgr: np.ndarray, quad: np.ndarray, out_w: int = 1000) -> np.ndarray:
    tl, tr, br, bl = quad
    max_w = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    max_h = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))
    out_h = int(max_h * out_w / max(1, max_w))
    dst = np.array([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]],
                   dtype=np.float32)
    return cv2.warpPerspective(bgr, cv2.getPerspectiveTransform(quad, dst), (out_w, out_h))


# ── Orientation ───────────────────────────────────────────────────────────────

def is_plausible_front_card_image(bgr: np.ndarray) -> bool:
    h, w = bgr.shape[:2]
    return 0.50 <= h / max(1, w) <= 0.85

def normalize_front_orientation(bgr: np.ndarray, out_w: int = 1000) -> tuple[np.ndarray, int]:
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    blue_mask = cv2.inRange(hsv, np.array([85, 45, 20]), np.array([135, 255, 180]))
    bh = max(1, int(blue_mask.shape[0] * 0.22))
    bw = max(1, int(blue_mask.shape[1] * 0.22))
    scores = {
        "top":    float((blue_mask[:bh] > 0).mean()),
        "bottom": float((blue_mask[-bh:] > 0).mean()),
        "left":   float((blue_mask[:, :bw] > 0).mean()),
        "right":  float((blue_mask[:, -bw:] > 0).mean()),
    }
    side = max(scores, key=scores.get)
    rot = 0
    if scores[side] > 0.03:
        if side == "bottom": bgr = cv2.rotate(bgr, cv2.ROTATE_180);             rot = 180
        elif side == "left":  bgr = cv2.rotate(bgr, cv2.ROTATE_90_CLOCKWISE);    rot = 90
        elif side == "right": bgr = cv2.rotate(bgr, cv2.ROTATE_90_COUNTERCLOCKWISE); rot = 270
    return normalize_width(bgr, out_w=out_w), rot

def normalize_card_geometry(bgr: np.ndarray, out_w: int = 1000) -> np.ndarray:
    bgr = normalize_width(bgr, out_w=out_w)
    bgr, _ = deskew(bgr, min_abs=0.20)
    quad = find_document_quad(bgr)
    return warp_document(bgr, quad, out_w=out_w) if quad is not None else normalize_width(bgr, out_w=out_w)


# ── Preprocessing ─────────────────────────────────────────────────────────────

def preprocess_for_ocr(bgr: np.ndarray) -> dict[str, np.ndarray]:
    gray    = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    denoise = cv2.fastNlMeansDenoising(clahe, None, h=10, templateWindowSize=7, searchWindowSize=21)
    sharpen = cv2.filter2D(denoise, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32))
    thresh  = cv2.adaptiveThreshold(sharpen, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY, 35, 10)
    return {"bgr": bgr, "gray": gray, "clahe": clahe,
            "denoise": denoise, "sharpen": sharpen, "thresh": thresh}


# ── ROI definitions ───────────────────────────────────────────────────────────
# Fractions relative to the warped image (out_w=1000, aspect ≈ 0.63).

@dataclass
class ROI:
    name: str
    x1: float; y1: float; x2: float; y2: float

FRONT_ROIS = [
    ROI("full_name",           0.510, 0.200, 0.925, 0.265),
    ROI("id_number",           0.510, 0.285, 0.925, 0.330),
    ROI("date_of_birth",       0.510, 0.355, 0.725, 0.400),
    ROI("sex",                 0.510, 0.425, 0.635, 0.470),
    ROI("nationality",         0.510, 0.495, 0.775, 0.540),
    ROI("place_of_origin",     0.510, 0.565, 0.925, 0.610),
    ROI("place_of_residence",  0.510, 0.630, 0.965, 0.690),
    ROI("issue_date",          0.510, 0.700, 0.710, 0.745),
    ROI("expiry_date",         0.510, 0.765, 0.710, 0.810),
]

def scale_rois(rois: list[ROI], w: int, h: int) -> dict[str, tuple[int, int, int, int]]:
    return {r.name: (int(r.x1*w), int(r.y1*h), int(r.x2*w), int(r.y2*h)) for r in rois}


# ── Text helpers ──────────────────────────────────────────────────────────────

_WATERMARK_PATTERNS = [
    r"\bNOT\s+A\s+REAL\s+ID\b", r"\bREAL\s+ID\b", r"\bEAL\s*ID\b",
    r"\bEALID\b", r"\bRALID\b", r"\bALID\b", r"\bDEMO\b", r"\bSAMPLE\b",
]

def clean_text(text: str) -> str:
    if not text: return ""
    text = unicodedata.normalize("NFC", str(text).strip())
    return re.sub(r"\s+", " ", re.sub(r"[`']", "", text.replace("|", "I"))).strip()

def strip_watermark(text: str) -> str:
    for p in _WATERMARK_PATTERNS:
        text = re.sub(p, " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

def _accentless(text: str) -> str:
    decomposed = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in decomposed if unicodedata.category(ch) != "Mn"
                   ).replace("đ", "d").replace("Đ", "D").lower()

def _drop_watermark_tokens(text: str) -> str:
    kept = []
    for tok in clean_text(text).split():
        key = _accentless(tok).upper()
        if re.fullmatch(r"[HILN]{3,}", key) or re.fullmatch(r"(?:EALID|RALID|ALID)", key):
            continue
        kept.append(tok)
    return " ".join(kept).strip()

def parse_field(field: str, text: str) -> str:
    text = clean_text(text)
    if not text: return ""
    if field in {"full_name", "place_of_origin", "place_of_residence"}:
        text = strip_watermark(text)
    key = _accentless(text)
    if field == "nationality" and "vietnam" in key.replace(" ", ""):
        return "Vietnam"
    if field == "sex":
        if re.search(r"\b(?:male|m)\b", key): return "Male"
        if re.search(r"\b(?:female|f)\b", key): return "Female"
    if field == "full_name":
        text = _drop_watermark_tokens(text)
        return " ".join(t[:1].upper() + t[1:].lower() for t in text.split())
    if field in {"place_of_origin", "place_of_residence"}:
        return re.sub(r"\s+", " ", text).strip()
    return text

def is_plausible(field: str, text: str) -> bool:
    text = clean_text(text)
    if not text: return False
    key = _accentless(text)
    if field == "id_number":
        return bool(re.search(r"(?:demo[-\s]*)?\d{6,}", text, flags=re.IGNORECASE))
    if field in {"date_of_birth", "issue_date", "expiry_date"}:
        return bool(re.search(r"\d{1,2}[/\-.]\d{1,2}[/\-.]\d{4}", text))
    if field == "sex":
        return bool(re.search(r"\b(?:male|female|m|f)\b", key))
    if field == "nationality":
        return "vietnam" in key.replace(" ", "")
    if field in {"full_name", "place_of_origin", "place_of_residence"}:
        return len(text) >= 2
    return True


# ── OCR utilities ─────────────────────────────────────────────────────────────

@dataclass
class OCRBox:
    text: str; score: float
    x1: float; y1: float; x2: float; y2: float
    @property
    def cx(self) -> float: return (self.x1 + self.x2) / 2.0
    @property
    def cy(self) -> float: return (self.y1 + self.y2) / 2.0
    @property
    def h(self) -> float: return max(1.0, self.y2 - self.y1)

def _box_to_xyxy(box) -> tuple | None:
    arr = np.asarray(box, dtype=np.float32)
    if arr.size == 4 and arr.ndim == 1: return tuple(arr.tolist())
    if arr.ndim >= 2 and arr.shape[-1] >= 2:
        pts = arr.reshape(-1, arr.shape[-1])[:, :2]
        return (float(np.min(pts[:, 0])), float(np.min(pts[:, 1])),
                float(np.max(pts[:, 0])), float(np.max(pts[:, 1])))
    return None

def ocr_predict_text(img: np.ndarray) -> tuple[str, float | None]:
    bgr_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if img.ndim == 2 else img
    results = ocr.predict(bgr_img, use_doc_orientation_classify=False,
                          use_doc_unwarping=False, use_textline_orientation=False)
    if not results: return "", None
    j = getattr(results[0], "json", None)
    if not isinstance(j, dict): return "", None
    res    = j.get("res", {})
    texts  = res.get("rec_texts",  []) or []
    scores = [float(s) for s in (res.get("rec_scores") or []) if s is not None]
    return " ".join(str(t) for t in texts).strip(), (float(np.mean(scores)) if scores else None)

def ocr_predict_boxes(img: np.ndarray) -> list[OCRBox]:
    bgr_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if img.ndim == 2 else img
    results = ocr.predict(bgr_img, use_doc_orientation_classify=False,
                          use_doc_unwarping=False, use_textline_orientation=False)
    if not results: return []
    j = getattr(results[0], "json", None)
    if not isinstance(j, dict): return []
    res    = j.get("res", {})
    texts  = res.get("rec_texts",  []) or []
    scores = res.get("rec_scores", []) or []
    boxes  = next((res.get(k) for k in ("rec_boxes","rec_polys","dt_boxes","dt_polys") if res.get(k)), [])
    out: list[OCRBox] = []
    for i, text in enumerate(texts):
        if i >= len(boxes): continue
        xyxy = _box_to_xyxy(boxes[i])
        if xyxy is None: continue
        out.append(OCRBox(str(text).strip(),
                          float(scores[i] if i < len(scores) and scores[i] is not None else 0.0),
                          *xyxy))
    return out


# ── Label-anchored extraction ─────────────────────────────────────────────────

FIELD_LABELS = {
    "full_name":          ["FULL NAME"],
    "id_number":          ["DEMO ID NO", "ID NO"],
    "date_of_birth":      ["DATE OF BIRTH", "BIRTH"],
    "sex":                ["SEX"],
    "nationality":        ["NATIONALITY"],
    "place_of_origin":    ["PLACE OF ORIGIN", "ORIGIN"],
    "place_of_residence": ["PLACE OF RESIDENCE", "RESIDENCE"],
    "issue_date":         ["ISSUE DATE"],
    "expiry_date":        ["EXPIRY DATE", "EXPIRY"],
}

def _match_key(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", text.lower())

def _label_sim(text: str, label: str) -> float:
    tk, lk = _match_key(text), _match_key(label)
    if not tk or not lk: return 0.0
    return 1.0 if lk in tk else SequenceMatcher(None, tk, lk).ratio()

def _is_label(text: str) -> bool:
    return any(_label_sim(text, lbl) >= 0.78 for lbls in FIELD_LABELS.values() for lbl in lbls)

def _is_watermark(text: str) -> bool:
    key = _match_key(text)
    return any(p in key for p in ["notarealid", "realid", "sample", "alid"])

def _find_label_box(boxes: list[OCRBox], field: str) -> OCRBox | None:
    best, best_score = None, 0.0
    for box in boxes:
        score = max(_label_sim(box.text, lbl) for lbl in FIELD_LABELS[field])
        if score > best_score: best, best_score = box, score
    return best if best_score >= 0.68 else None

def _voverlap(a: OCRBox, b: OCRBox) -> float:
    return max(0.0, min(a.y2, b.y2) - max(a.y1, b.y1)) / max(1.0, min(a.h, b.h))

def extract_by_labels(boxes: list[OCRBox]) -> tuple[dict[str, str], dict[str, float]]:
    if not boxes: return {}, {}
    median_h = float(np.median([b.h for b in boxes]))
    raw: dict[str, str] = {}
    scores: dict[str, float] = {}
    for field in FIELD_LABELS:
        label_box = _find_label_box(boxes, field)
        if label_box is None: continue
        candidates = [
            b for b in boxes
            if b is not label_box and not _is_label(b.text) and not _is_watermark(b.text)
            and b.x1 >= label_box.x2 - 8
            and (_voverlap(label_box, b) >= 0.20
                 or abs(b.cy - label_box.cy) <= max(label_box.h, b.h, median_h) * 0.85)
        ]
        if not candidates: continue
        candidates.sort(key=lambda b: (b.x1, b.y1))
        if field == "id_number":
            candidates = [b for b in candidates
                          if re.search(r"(?:demo[-\s]*)?\d{6,}", b.text, re.IGNORECASE)][:1]
        elif field in {"date_of_birth", "issue_date", "expiry_date"}:
            candidates = [b for b in candidates
                          if re.search(r"\d{1,2}[/\-.]\d{1,2}[/\-.]\d{4}", b.text)][:1]
        if not candidates: continue
        text = " ".join(b.text for b in candidates).strip()
        if text:
            raw[field]    = text
            scores[field] = float(np.mean([b.score for b in candidates]))
    return raw, scores


# ── ROI fallback for full_name ────────────────────────────────────────────────

def _binary_for_ocr(img: np.ndarray) -> np.ndarray:
    if img.ndim != 2: return img
    return cv2.bitwise_not(img) if float(img.mean()) < 127.0 else img

def _name_roi_variants(pp: dict, box: tuple[int, int, int, int]) -> list[tuple[str, np.ndarray]]:
    x1, y1, x2, y2 = box
    out: list[tuple[str, np.ndarray]] = []
    gray = pp.get("gray")
    if gray is not None:
        roi = gray[y1:y2, x1:x2]
        if roi.size:
            out.append(("gray", roi))
            blur = cv2.GaussianBlur(roi, (3, 3), 0)
            _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            out.append(("gray_otsu", _binary_for_ocr(th)))
    for key in ("denoise", "sharpen", "clahe"):
        base = pp.get(key)
        if base is not None:
            roi = base[y1:y2, x1:x2]
            if roi.size: out.append((key, roi))
    seen: set[str] = set()
    return [(k, v) for k, v in out if k not in seen and not seen.add(k)]  # type: ignore[func-returns-value]

def _choose_best(field: str, candidates: list[tuple[str, str, float | None]]) -> tuple[str, float | None, str]:
    best_text, best_score, best_src = "", None, ""
    best_key = (-1, -1, -1.0, -1)
    for src, raw_text, score in candidates:
        cleaned = strip_watermark(clean_text(raw_text)) \
            if field in {"full_name", "place_of_origin", "place_of_residence"} \
            else clean_text(raw_text)
        corrected = parse_field(field, cleaned)
        plaus  = 1 if is_plausible(field, corrected) else 0
        alpha  = sum(ch.isalpha() for ch in corrected)
        s      = float(score) if score is not None else -1.0
        key    = (plaus, alpha, s, len(corrected))
        if key > best_key:
            best_key = key; best_text = raw_text; best_score = score; best_src = src
    return best_text, best_score, best_src

def ocr_name_fallback(pp: dict, box: tuple[int, int, int, int]) -> tuple[str, float | None, str]:
    candidates: list[tuple[str, str, float | None]] = []
    for src, roi in _name_roi_variants(pp, box):
        t, s = ocr_predict_text(roi)
        candidates.append((src, t, s))
        for scale in (2.0, 3.0):
            try:
                up = cv2.resize(roi, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
                t2, s2 = ocr_predict_text(up)
                candidates.append((f"{src}_x{int(scale)}", t2, s2))
            except Exception:
                pass
    return _choose_best("full_name", candidates)


print("Helpers loaded — FRONT_ROIS fields:", [r.name for r in FRONT_ROIS])

## Step 1 – Load Images
Load ảnh đầu vào và in thông tin cơ bản của từng file.

In [ ]:
# Step 1: Load Images
input_images = []
input_rows = []

for path in img_paths:
    bgr = cv2.imread(str(path))
    status = "ok" if bgr is not None else "read_error"
    row = {"file": path.name, "file_path": str(path),
           "user_id": extract_user_id(path), "status": status}
    if bgr is not None:
        row.update({"height": bgr.shape[0], "width": bgr.shape[1]})
        input_images.append({"path": path, "bgr": bgr, **row})
    input_rows.append(row)

print("=== STEP 1: LOAD IMAGES ===")
print(f"Loaded {len(input_images)}/{len(img_paths)} images")
df_input = pd.DataFrame(input_rows)
display(df_input)

for item in input_images[:PREVIEW_N]:
    show_bgr(f"Input – user_id={item['user_id']} – {item['file']}", item["bgr"], max_w=9)

## Step 2 – Preprocess
Tiền xử lý ảnh qua 6 bước nhỏ, mỗi bước in/hiển thị kết quả trung gian (cho `PREVIEW_N` ảnh đầu):

1. **Deskew** — ước lượng góc nghiêng bằng Hough lines rồi xoay bù.
2. **Detect quad & warp** — tìm 4 góc thẻ từ contour, warp phối cảnh về mặt phẳng.
3. **Orientation** — dựa vào dải màu xanh quốc huy để xoay đúng chiều (0/90/180/270°).
4. **Geometry** — chuẩn hóa hình học lần cuối (deskew nhẹ + warp lại nếu cần).
5. **Biến thể OCR** — gray / CLAHE / denoise / sharpen / threshold.
6. **ROI layout** — định vị vùng từng field theo tỉ lệ trên ảnh đã warp.

In [ ]:
# Step 2: Preprocess
# Quy trình: 2.1 deskew → 2.2 tìm quad & warp → 2.3 chuẩn hóa hướng
#            → 2.4 chuẩn hóa hình học → 2.5 biến thể ảnh cho OCR → 2.6 định vị ROI
preprocessed_docs = []
preprocess_rows = []

for idx, item in enumerate(input_images):
    bgr = item["bgr"]
    verbose = idx < PREVIEW_N          # chỉ vẽ minh họa cho vài ảnh đầu để đỡ rối output

    if verbose:
        print(f"\n────── user_id={item['user_id']} – {item['file']} ──────")

    # ── 2.1 Deskew: ước lượng góc nghiêng (Hough lines trên cạnh) và xoay bù ──
    deskewed, deskew_deg = deskew(bgr)
    if verbose:
        print(f"[2.1] Deskew      : góc bù = {deskew_deg:+.2f}°"
              + ("  (ảnh đã thẳng, không cần xoay)" if deskew_deg == 0.0 else ""))
        if deskew_deg != 0.0:
            show_bgr(f"2.1 Sau deskew ({deskew_deg:+.2f}°)", deskewed, max_w=7)

    # ── 2.2 Tìm 4 góc thẻ (contour → quad) rồi warp phối cảnh về mặt phẳng ──
    quad = find_document_quad(deskewed)
    if quad is not None:
        warped = warp_document(deskewed, quad, out_w=1000)
        if verbose:
            print(f"[2.2] Detect quad : tìm thấy 4 góc thẻ → warp phối cảnh")
            overlay = deskewed.copy()
            cv2.polylines(overlay, [quad.astype(np.int32)], True, (0, 0, 255), 4)
            show_bgr("2.2 Quad phát hiện (viền đỏ)", overlay, max_w=7)
    else:
        warped = normalize_width(deskewed, out_w=1000)
        if verbose:
            print("[2.2] Detect quad : KHÔNG tìm thấy → chỉ resize về bề rộng 1000px")

    # ── 2.3 Chuẩn hóa hướng: dò dải màu xanh (quốc huy/header) để xoay 0/90/180/270° ──
    warped, rot_deg = normalize_front_orientation(warped, out_w=1000)
    if not is_plausible_front_card_image(warped):
        # tỉ lệ H/W ngoài khoảng thẻ CCCD → warp hỏng, quay về xử lý từ ảnh gốc
        warped, rot_deg = normalize_front_orientation(bgr, out_w=1000)
        if verbose:
            print("[2.3] Orientation : kết quả warp không hợp lệ → dùng lại ảnh gốc")
    if verbose:
        print(f"[2.3] Orientation : xoay {rot_deg}°")

    # ── 2.4 Chuẩn hóa hình học lần cuối (deskew nhẹ + warp lại nếu tìm được quad) ──
    warped = normalize_card_geometry(warped, out_w=1000)
    if verbose:
        print(f"[2.4] Geometry    : kích thước cuối = {warped.shape[1]}×{warped.shape[0]}")
        show_bgr("2.4 Ảnh thẻ sau chuẩn hóa (đầu vào OCR)", warped, max_w=8)

    # ── 2.5 Sinh các biến thể tiền xử lý cho OCR (dùng cho fallback ở Step 3) ──
    pp = preprocess_for_ocr(warped)
    if verbose:
        variants = ["gray", "clahe", "denoise", "sharpen", "thresh"]
        fig, axes = plt.subplots(1, len(variants), figsize=(3.4 * len(variants), 2.8))
        for ax, key in zip(np.atleast_1d(axes), variants):
            ax.imshow(pp[key], cmap="gray"); ax.set_title(f"2.5 {key}"); ax.axis("off")
        plt.tight_layout(); plt.show()

    # ── 2.6 Định vị ROI từng field theo tỉ lệ cố định trên ảnh đã warp ──
    roi_boxes = scale_rois(FRONT_ROIS, warped.shape[1], warped.shape[0])
    if verbose:
        roi_vis = warped.copy()
        for name, (x1, y1, x2, y2) in roi_boxes.items():
            cv2.rectangle(roi_vis, (x1, y1), (x2, y2), (0, 200, 0), 2)
            cv2.putText(roi_vis, name, (x1, max(12, y1 - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 120, 255), 1, cv2.LINE_AA)
        show_bgr("2.6 Layout ROI các field (khung xanh)", roi_vis, max_w=8)

    preprocessed_docs.append({**item, "warped": warped, "pp": pp, "roi_boxes": roi_boxes})
    preprocess_rows.append({
        "file":        item["file"],
        "user_id":     item["user_id"],
        "deskew_deg":  round(deskew_deg, 2),
        "rotation":    rot_deg,
        "quad_found":  quad is not None,
        "warped_size": f"{warped.shape[1]}×{warped.shape[0]}",
    })

print("\n=== STEP 2: PREPROCESS – BẢNG TỔNG HỢP ===")
df_preprocess = pd.DataFrame(preprocess_rows)
display(df_preprocess)

## Step 3 – OCR
Nhận dạng text qua 3 bước nhỏ, in kết quả trung gian từng bước:

1. **OCR toàn ảnh** — PaddleOCR đọc mọi dòng text kèm tọa độ box + score.
2. **Label-anchored extraction** — tìm box chứa label (FULL NAME, ID NO, …) rồi lấy text bên phải làm giá trị field.
3. **ROI fallback** — field nào chưa hợp lệ thì OCR lại trên vùng ROI đã crop (riêng `full_name` thử nhiều biến thể ảnh + phóng to).

In [ ]:
# Step 3: OCR
# Quy trình: 3.1 OCR toàn ảnh → 3.2 neo theo label → 3.3 fallback ROI cho field chưa hợp lệ
ocr = PaddleOCR(lang=OCR_LANG, ocr_version="PP-OCRv3", use_textline_orientation=False)

recognized_docs = []
recognition_rows = []

for idx, item in enumerate(preprocessed_docs):
    warped    = item["warped"]
    pp        = item["pp"]
    roi_boxes = item["roi_boxes"]
    verbose   = idx < PREVIEW_N

    if verbose:
        print(f"\n────── user_id={item['user_id']} – {item['file']} ──────")

    # ── 3.1 OCR toàn ảnh: lấy mọi dòng text + tọa độ box + score ──
    text_boxes = ocr_predict_boxes(warped)
    if verbose:
        print(f"[3.1] OCR toàn ảnh: đọc được {len(text_boxes)} dòng text")
        for b in sorted(text_boxes, key=lambda b: (b.y1, b.x1)):
            print(f"      (x={b.x1:4.0f}, y={b.y1:4.0f})  score={b.score:.3f}  {b.text!r}")
        box_vis = warped.copy()
        for b in text_boxes:
            cv2.rectangle(box_vis, (int(b.x1), int(b.y1)), (int(b.x2), int(b.y2)),
                          (255, 0, 0), 2)
        show_bgr("3.1 Các box text PaddleOCR phát hiện (khung xanh dương)", box_vis, max_w=8)

    # ── 3.2 Trích xuất neo theo label: tìm box label rồi lấy text bên phải ──
    anchored_raw, anchored_scores = extract_by_labels(text_boxes)
    if verbose:
        print(f"[3.2] Label-anchored: bắt được {len(anchored_raw)}/{len(FIELD_LABELS)} field")
        for fn, txt in anchored_raw.items():
            print(f"      {fn:22s}: {txt!r}  (score={anchored_scores.get(fn, 0):.3f})")

    # ── 3.3 Fallback ROI: field nào parse ra chưa hợp lệ thì OCR lại trên vùng crop ──
    field_raw: dict[str, dict] = {}
    for field_name, box in roi_boxes.items():
        raw   = anchored_raw.get(field_name, "")
        score = anchored_scores.get(field_name)
        src   = "label_anchored"

        if not is_plausible(field_name, parse_field(field_name, raw)):
            if field_name == "full_name":
                # full_name khó đọc → thử nhiều biến thể tiền xử lý + phóng to, chọn ứng viên tốt nhất
                raw, score, variant = ocr_name_fallback(pp, box)
                src = f"roi_fallback[{variant}]"
            else:
                roi_img = pp["clahe"][box[1]:box[3], box[0]:box[2]]
                raw, score = ocr_predict_text(roi_img)
                src = "roi_fallback"
            if verbose:
                print(f"[3.3] Fallback     : {field_name:22s} → {raw!r}  [{src}]")

        field_raw[field_name] = {"raw_text": raw, "score": score, "src": src}
        recognition_rows.append({
            "file":     item["file"],
            "user_id":  item["user_id"],
            "field":    field_name,
            "raw_text": raw,
            "score":    round(score, 4) if score is not None else None,
            "src":      src,
        })

    recognized_docs.append({**item, "field_raw": field_raw})

print("\n=== STEP 3: OCR – BẢNG TỔNG HỢP (raw text từng field) ===")
df_recognition = pd.DataFrame(recognition_rows)
display(df_recognition)

for item in recognized_docs[:PREVIEW_N]:
    print(f"\nField preview – user_id={item['user_id']} – {item['file']}")
    for fn, payload in item["field_raw"].items():
        print(f"  {fn:22s}: {payload['raw_text']!r}  [{payload['src']}]")

## Step 4 – Parse & Structure
Chuẩn hóa từng field bằng `parse_field` (bỏ watermark "NOT A REAL ID", chuẩn hoa/thường tên, map `sex`/`nationality` về giá trị chuẩn) rồi gom thành record có cấu trúc.

In hai bảng trung gian: **raw → parsed** (chỉ những dòng parse có biến đổi, kèm cờ `plausible`) và bảng record cuối.

In [ ]:
# Step 4: Parse & Structure
# Chuẩn hóa từng field (bỏ watermark, sửa hoa/thường, map giới tính/quốc tịch...)
# và in bảng so sánh raw → parsed để thấy rõ từng phép biến đổi.
structured_docs = []
structured_rows = []
parse_compare_rows = []

for item in recognized_docs:
    parsed_fields = {}
    for fn, payload in item["field_raw"].items():
        raw    = payload["raw_text"]
        parsed = parse_field(fn, raw)
        parsed_fields[fn] = parsed
        parse_compare_rows.append({
            "file":      item["file"],
            "user_id":   item["user_id"],
            "field":     fn,
            "raw_text":  raw,
            "parsed":    parsed,
            "changed":   clean_text(raw) != parsed,            # parse có biến đổi text không
            "plausible": is_plausible(fn, parsed),             # giá trị sau parse có hợp lệ không
        })

    structured_docs.append({**item, "parsed_fields": parsed_fields})
    structured_rows.append({
        "file":               item["file"],
        "file_path":          str(item["path"]),
        "run_date":           RUN_DATE,
        "user_id":            item["user_id"],
        "full_name":          parsed_fields.get("full_name"),
        "id_number":          parsed_fields.get("id_number"),
        "date_of_birth":      parsed_fields.get("date_of_birth"),
        "sex":                parsed_fields.get("sex"),
        "nationality":        parsed_fields.get("nationality"),
        "place_of_origin":    parsed_fields.get("place_of_origin"),
        "place_of_residence": parsed_fields.get("place_of_residence"),
        "issue_date":         parsed_fields.get("issue_date"),
        "expiry_date":        parsed_fields.get("expiry_date"),
    })

print("=== STEP 4: PARSE – SO SÁNH RAW → PARSED (chỉ các dòng có biến đổi) ===")
df_parse_compare = pd.DataFrame(parse_compare_rows)
display(df_parse_compare[df_parse_compare["changed"]])

print("=== STEP 4: RECORD CÓ CẤU TRÚC ===")
df_structured = pd.DataFrame(structured_rows)
display(df_structured)

## Step 5 – Confidence & Save
Tính confidence theo OCR score trung bình và tỉ lệ field plausible; merge vào kết quả cuối.

In [ ]:
# Step 5: Confidence Scoring
# final_confidence = 0.70 × (OCR score trung bình) + 0.30 × (tỉ lệ field plausible)
confidence_rows = []

for item in structured_docs:
    scores = [float(p["score"]) for p in item["field_raw"].values() if p.get("score") is not None]
    plaus  = sum(1 for fn, pf in item["parsed_fields"].items() if is_plausible(fn, pf))
    total  = len(item["field_raw"])
    bad_fields = [fn for fn, pf in item["parsed_fields"].items() if not is_plausible(fn, pf)]
    ocr_conf   = float(np.mean(scores)) if scores else 0.0
    parse_conf = plaus / max(1, total)
    final_conf = round(0.70 * ocr_conf + 0.30 * parse_conf, 4)
    confidence_rows.append({
        "file":             item["file"],
        "user_id":          item["user_id"],
        "ocr_confidence":   round(ocr_conf, 4),
        "parse_confidence": round(parse_conf, 4),
        "plausible_fields": plaus,
        "total_fields":     total,
        "bad_fields":       ", ".join(bad_fields) if bad_fields else "",
        "final_confidence": final_conf,
    })

print("=== STEP 5: CONFIDENCE ===")
print("Công thức: final = 0.70 × ocr_confidence + 0.30 × parse_confidence")
df_confidence = pd.DataFrame(confidence_rows)
display(df_confidence)

# Cảnh báo các ảnh có field chưa hợp lệ để dễ rà soát
for row in confidence_rows:
    if row["bad_fields"]:
        print(f"  ⚠ user_id={row['user_id']} – {row['file']}: field chưa hợp lệ → {row['bad_fields']}")

df_final = df_structured.merge(
    df_confidence[["file", "user_id", "final_confidence", "ocr_confidence",
                   "parse_confidence", "plausible_fields"]],
    on=["file", "user_id"], how="left",
)
print("\n=== FINAL RESULT ===")
display(df_final)

In [ ]:
# Save extraction result
OUT_CSV = PROJECT_ROOT / "data" / "unstructured" / "extracted" / f"id_card_extractions_{RUN_DATE}.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUT_CSV}")